# Colab Run All - Google Drive Workflow

Use this single notebook on a Google Colab T4 runtime. It clones or updates the GitHub repo, mounts Google Drive, installs Colab-safe dependencies, copies private metadata from Drive into the ignored local data folder, runs sanity checks, and verifies the project code.

In [ ]:
REPO_URL = "https://github.com/SalmaneSossey/graphcnn-federated-3d.git"
REPO_DIR = "/content/graphcnn-federated-3d"
BRANCH = "main"

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/graphcnn-federated-3d"
DRIVE_METADATA_CSV = f"{DRIVE_PROJECT_DIR}/labeled_dataset.csv"

DOWNLOAD_CAP3D_FILES = False

print("Repository:", REPO_URL)
print("Colab repo directory:", REPO_DIR)
print("Google Drive project directory:", DRIVE_PROJECT_DIR)
print("Drive metadata CSV:", DRIVE_METADATA_CSV)

## Mount Drive

Put your private `labeled_dataset.csv` at `MyDrive/graphcnn-federated-3d/labeled_dataset.csv`. The notebook copies it into the Colab runtime only. It remains ignored by Git.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Clone or Update Repo

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

def run(command, check=True):
    print("$", " ".join(str(part) for part in command))
    return subprocess.run(command, check=check)

repo_path = Path(REPO_DIR)
if repo_path.exists():
    os.chdir(REPO_DIR)
    run(["git", "fetch", "origin"])
    run(["git", "checkout", BRANCH])
    run(["git", "pull", "--ff-only", "origin", BRANCH])
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

## Install Dependencies

This uses `requirements-colab.txt`, which intentionally does not reinstall PyTorch because Colab already provides a CUDA-enabled build.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements-colab.txt

## GPU Check

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!nvidia-smi || true

## Copy Private Metadata

Google Drive is the private data source. GitHub remains code-only.

In [ ]:
drive_project = Path(DRIVE_PROJECT_DIR)
drive_project.mkdir(parents=True, exist_ok=True)

for folder in ["data/metadata", "data/raw", "data/processed", "data/splits", "data/cache", "checkpoints", "outputs", "runs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

repo_metadata = Path("data/metadata/labeled_dataset.csv")
drive_metadata = Path(DRIVE_METADATA_CSV)

if drive_metadata.exists():
    shutil.copy2(drive_metadata, repo_metadata)
    print(f"Copied metadata from Drive to {repo_metadata}")
else:
    print(f"Metadata not found. Upload it to: {drive_metadata}")
    print("The notebook will still run code tests, but data checks will be skipped.")

## Optional Cap3D Download

This is off by default. Set `DOWNLOAD_CAP3D_FILES = True` in the first cell only when you are ready. Do not unzip the full archive blindly.

In [ ]:
if DOWNLOAD_CAP3D_FILES:
    !python scripts/download_cap3d.py --data-dir data/raw
else:
    print("Skipped Cap3D download. Set DOWNLOAD_CAP3D_FILES=True when needed.")

## Metadata Sanity Check

In [ ]:
import pandas as pd

from src.data.dataset import ShapeNetPointCloudDataset
from src.utils.config import load_config

config = load_config("configs/data.yaml")
metadata_path = Path(config["paths"]["metadata_csv"])

if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head())
else:
    df = pd.DataFrame()
    print(f"Missing metadata: {metadata_path}")

In [ ]:
if not df.empty:
    label_candidates = [c for c in df.columns if "label" in c.lower() or "class" in c.lower() or "category" in c.lower()]
    label_col = "label" if "label" in df.columns else (label_candidates[0] if label_candidates else None)
    print("Label column:", label_col)
    if label_col:
        counts = df[label_col].value_counts()
        display(counts.head(30).rename_axis("label").reset_index(name="count"))

        subset_classes = int(config["subset_classes"])
        samples_per_class = int(config["samples_per_class"])
        chosen_labels = counts.head(subset_classes).index.tolist()
        subset_preview = (
            df[df[label_col].isin(chosen_labels)]
            .groupby(label_col, group_keys=False)
            .head(samples_per_class)
            .reset_index(drop=True)
        )
        print("Chosen labels:", chosen_labels)
        print("Subset preview rows:", len(subset_preview))
        display(subset_preview[label_col].value_counts().rename_axis("label").reset_index(name="count"))
else:
    print("Metadata sanity check skipped until labeled_dataset.csv is available.")

## Code Verification

In [ ]:
dataset = ShapeNetPointCloudDataset(metadata_csv="data/metadata/labeled_dataset.csv")
print(dataset.summary())

!pytest -q
!python main.py
!python scripts/train_pointgcn_centralized.py
!python scripts/run_vfl.py